In [46]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [47]:
from rag_helper import RAG
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

Create the assistant

In [48]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering thhe QUESTION.
""".strip()

assistant = RAG(
    index=index,
    llm_client=openai_client,
    instructions=instructions
)

In [49]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your operating system:
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and start a model:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. To check that the local server is running, use:
   ```bash
   curl http://localhost:11434
   ```
   You should get a response like:
   ```json
   {"models": [...]}  
   ```

If you need to restart the Ollama server while working in the notebook, run:
```bash
!nohup ollama serve > nohup.out 2>&1 &
```


In [50]:
answer = assistant.rag('How can I run olama locally?')

print(answer)

You can run a model locally with **Ollama**. The course says that locally served models like Ollama work without an external API call, and that most of these expose an **OpenAI-compatible endpoint**, so the course code usually works with only a **`base_url` change**.


In [51]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I stil join the course?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages
)

response.output_text

'Here’s a polished version of your message:\n\n**“I just discovered the course. Can I still join?”**\n\nIf you want, I can also help make it sound more **polite**, **formal**, or **friendly** depending on who you’re writing to.'

In [52]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [53]:
index.search('How I use Ollama?')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [54]:
# Function definition in JSON

search_tool = {
    'type': 'function',
    'name': 'search',
    'description': 'Search the FAQ for entries about the given query.',
    'parameters': {
        'type': 'object',
        'properties': {
            'query': {
                'type': 'string',
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        'reuired': ['query'],
        'additionalProperties': False
    }
}

In [55]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [56]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I still join the course? late enrollment discovered course join after start"}', call_id='call_ihhfHL0hU7oauNSrHCB9fX6Z', name='search', type='function_call', id='fc_04266da86303037c006a7c7f49d8548194886ac9551f0c1ae3', caller=None, namespace=None, status='completed')]

In [57]:
call = response.output[0]

In [58]:
import json

args = json.loads(call.arguments)

In [59]:
call.name

'search'

In [60]:
results = search(**args)

In [61]:
result_json = json.dumps(results, indent=2)

In [62]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

In [63]:
function_call_output = {
    'type': 'function_call_output',
    'call_id': call.call_id,
    'output': result_json
}

print(function_call_output)

{'type': 'function_call_output', 'call_id': 'call_ihhfHL0hU7oauNSrHCB9fX6Z', 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\\n\\nYou can start whenever you want. The videos and GitHub materials are available, and the dead

In [64]:
messages.append(function_call_output)

In [65]:
messages

[{'role': 'user',
  'content': 'I just discovered the course. Can I stil join the course?'},
 {'type': 'function_call_output',
  'call_id': 'call_ihhfHL0hU7oauNSrHCB9fX6Z',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).

In [66]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

BadRequestError: Error code: 400 - {'error': {'message': 'No tool call found for function call output with call_id call_ihhfHL0hU7oauNSrHCB9fX6Z.', 'type': 'invalid_request_error', 'param': 'input', 'code': None}}